# 粒子群算法实验：校园 WiFi 路由器布置位置优化（学生练习版）

本 Notebook 是学生练习版：教学区场景建模、环境可视化、信号强度估计和目标函数已经完整给出，不作为考查内容；粒子群算法优化器中的关键步骤被标记为 `TODO`，需要学生补全。

本实验使用 **粒子群算法 Particle Swarm Optimization, PSO** 优化校园 WiFi 路由器布置位置。假设学校准备在若干校园区域部署一批 WiFi 路由器，需要选择合适的安装坐标，使得重点区域的无线覆盖尽可能好，同时减少弱信号、路由器互相干扰和不合理安装位置。

本实验重点展示：

1. 校园 WiFi 路由器布置问题与路径选择问题的区别。
2. 如何把“覆盖、信号、障碍物、干扰、负载”转化为可计算的优化目标。
3. 如何为宿舍区、教学区、图书馆广场区设计不同需求场景。
4. 如何使用粒子群算法搜索较优的路由器安装位置。
5. 如何可视化最优布置方案和算法收敛过程。

## 1. WiFi 路由器布置与路径选择的不同

校园路径选择问题通常关心“从哪里出发，按什么顺序到达哪些地点”。例如无人配送路径规划中，算法要决定快递站、宿舍楼、教学楼等节点的访问顺序，本质上更接近图搜索、旅行商问题或车辆路径规划。

WiFi 路由器布置问题则不同。它关心的是“把若干个路由器放在校园平面上的哪些位置”。每个解不是一条访问顺序，而是一组连续坐标，例如：

`[(x1, y1), (x2, y2), (x3, y3)]`

两类问题的主要区别如下：

| 比较项 | 路径选择问题 | WiFi 路由器布置问题 |
|---|---|---|
| 决策对象 | 节点访问顺序或道路选择 | 多个路由器的安装坐标 |
| 解的形式 | 离散序列，例如 `[0, 4, 2, 1, 0]` | 连续坐标，例如 `[(20, 30), (80, 60)]` |
| 优化目标 | 路径总代价最小，如距离、拥堵、爬坡 | 覆盖质量最好，弱信号最少，干扰较低 |
| 约束因素 | 道路、路障、坡度、交通拥堵 | 墙体遮挡、禁装区域、覆盖半径、用户密度 |
| 常用算法 | A*、遗传算法、蚁群算法等 | 粒子群算法、遗传算法、模拟退火等 |

因此，本实验使用粒子群算法解决一个连续空间中的位置优化问题，而不是求一条行走路线。

## 2. 路由器布置的要求

校园 WiFi 路由器布置需要综合考虑多个现实因素：

1. **覆盖范围**：宿舍、教室、图书馆座位区等重点区域应尽量被强信号覆盖。
2. **用户需求强度**：人员越密集、上网需求越高的区域权重越大。
3. **障碍物遮挡**：墙体、楼栋、隔断会造成信号衰减。
4. **禁装区域**：湖面、道路中央、施工区、绿化带等位置不能安装路由器。
5. **路由器干扰**：两个路由器距离过近会造成同频干扰，因此不宜过度集中。
6. **负载均衡**：不能让所有用户都集中连接同一个路由器，否则容易拥塞。
7. **部署成本**：路由器数量有限，位置应尽量服务更多需求点。

本实验将上述要求转化为一个综合目标函数。粒子群算法会不断调整路由器坐标，使得目标函数值尽可能小。

## 3. 实验设置：教学区 WiFi 布置场景

本实验只保留一个 **教学区场景**。教学区包含多个教学楼、实验楼、报告厅、教学广场和连廊休息区。该场景具有以下特点：

1. 上课时间用户密集，教学楼和报告厅需求权重较高。
2. 建筑墙体会造成明显信号衰减。
3. 施工禁装区不能安装 WiFi 路由器。
4. 路由器数量有限，需要在覆盖率、干扰和负载均衡之间折中。

场景包含：

- 校园区域大小。
- 一组带权重的 WiFi 需求点。
- 若干矩形障碍物或禁装区域。
- 可部署路由器数量。
- 信号覆盖阈值和干扰距离要求。

下面的程序会构造教学区场景，并基于粒子群算法选取较优路由器安装位置。

### 3.1 可修改实验参数

下面这个单元用于设置本实验中要部署的 WiFi 路由器数量。学生可以直接修改 `ROUTER_COUNT` 的值，例如改成 `3`、`5` 或 `6`，然后从本单元开始重新运行后面的代码。

注意：路由器数量越多，粒子群算法的搜索维度越高，因为每个路由器需要优化一个 `(x, y)` 坐标。

In [ ]:
# 可以在这里修改教学区要部署的 WiFi 路由器数量。
# 例如：ROUTER_COUNT = 3 表示部署 3 个路由器；ROUTER_COUNT = 5 表示部署 5 个路由器。
ROUTER_COUNT = 4
print(f"当前设置的路由器数量：{ROUTER_COUNT}")

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt

random.seed(42)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


@dataclass(frozen=True)
class DemandPoint:
    """WiFi 需求点。

    参数：
    - name：需求点名称。
    - x, y：需求点在校园平面中的坐标。
    - weight：需求强度，值越大表示该位置越重要。
    """
    name: str
    x: float
    y: float
    weight: float


@dataclass(frozen=True)
class Obstacle:
    """矩形障碍物或禁装区域。

    参数：
    - name：障碍物名称。
    - x1, y1, x2, y2：矩形左下角和右上角坐标。
    - loss：信号穿过该区域时的额外衰减。
    - no_install：是否禁止在该矩形内安装路由器。
    """
    name: str
    x1: float
    y1: float
    x2: float
    y2: float
    loss: float
    no_install: bool = True

    def contains(self, x: float, y: float) -> bool:
        return self.x1 <= x <= self.x2 and self.y1 <= y <= self.y2


@dataclass(frozen=True)
class CampusScenario:
    """校园 WiFi 优化场景。

    参数：
    - name：场景名称。
    - width, height：校园区域宽度和高度。
    - demands：需求点列表。
    - obstacles：障碍物或禁装区域列表。
    - router_count：需要部署的路由器数量。
    - signal_threshold：认为信号可用的最低强度。
    - min_router_distance：路由器之间建议保持的最小距离。
    """
    name: str
    width: float
    height: float
    demands: List[DemandPoint]
    obstacles: List[Obstacle]
    router_count: int
    signal_threshold: float = -58.0
    min_router_distance: float = 18.0


def make_teaching_scenario() -> CampusScenario:
    """构造教学区场景。"""
    demands = [
        DemandPoint("第一教学楼", 18, 25, 2.2),
        DemandPoint("第二教学楼", 38, 28, 2.0),
        DemandPoint("第三教学楼", 62, 26, 2.1),
        DemandPoint("实验楼", 82, 30, 1.8),
        DemandPoint("报告厅", 52, 58, 2.4),
        DemandPoint("教学广场", 48, 42, 1.7),
        DemandPoint("教师办公室", 75, 68, 1.4),
        DemandPoint("连廊休息区", 28, 56, 1.3),
    ]
    obstacles = [
        Obstacle("教学楼群 A", 10, 15, 45, 22, 9.0),
        Obstacle("教学楼群 B", 55, 15, 90, 23, 9.0),
        Obstacle("报告厅墙体", 42, 52, 64, 66, 10.0),
        Obstacle("施工禁装区", 28, 35, 38, 48, 7.0),
    ]
    return CampusScenario("教学区", 100, 85, demands, obstacles, router_count=ROUTER_COUNT, min_router_distance=16.0)


scenario = make_teaching_scenario()
scenarios = [scenario]

print(
    f"{scenario.name}: 需求点 {len(scenario.demands)} 个，"
    f"障碍物 {len(scenario.obstacles)} 个，路由器 {scenario.router_count} 个"
)

In [ ]:
def plot_scenario(scenario: CampusScenario) -> None:
    """绘制一个校园 WiFi 场景的需求点和障碍物。"""
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_title(f"{scenario.name} WiFi 布置场景")
    ax.set_xlim(0, scenario.width)
    ax.set_ylim(0, scenario.height)
    ax.set_xlabel("校园平面 x 坐标")
    ax.set_ylabel("校园平面 y 坐标")
    ax.grid(True, linestyle="--", alpha=0.3)

    for obstacle in scenario.obstacles:
        color = "#d9b38c" if obstacle.no_install else "#b5d6a7"
        rect = plt.Rectangle(
            (obstacle.x1, obstacle.y1),
            obstacle.x2 - obstacle.x1,
            obstacle.y2 - obstacle.y1,
            facecolor=color,
            edgecolor="#6d4c41",
            alpha=0.55,
        )
        ax.add_patch(rect)
        ax.text((obstacle.x1 + obstacle.x2) / 2, (obstacle.y1 + obstacle.y2) / 2,
                obstacle.name, ha="center", va="center", fontsize=8)

    for point in scenario.demands:
        size = 70 + point.weight * 55
        ax.scatter(point.x, point.y, s=size, color="#1976d2", edgecolor="white", zorder=3)
        ax.text(point.x + 1.5, point.y + 1.5, f"{point.name}\\n权重 {point.weight}", fontsize=8)

    ax.text(0.02, 0.02,
            f"路由器数量：{scenario.router_count}\\n最低信号阈值：{scenario.signal_threshold} dBm",
            transform=ax.transAxes,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#999999"})
    plt.show()


plot_scenario(scenario)

### 3.2 ????????????

下面的可视化程序用于更清楚地展示当前校园环境：

- 蓝色圆点表示 WiFi 需求热点，圆点越大表示需求强度越高。
- 红色斜线矩形表示 **无法安装 WiFi 路由器的区域**，例如湖面、施工区、绿化禁装区等。
- 棕色矩形表示建筑墙体或遮挡区域，会造成信号衰减。
- 绿色矩形表示可安装但仍会影响信号传播的建筑区域。

这一步不涉及粒子群算法，只帮助学生理解“优化算法面对的校园环境是什么样的”。

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def describe_obstacle_type(obstacle: Obstacle) -> str:
    """根据障碍物属性给出环境特点说明。"""
    if obstacle.no_install:
        return "禁装区域，不能放置路由器"
    return "可安装区域，但存在信号衰减"


def plot_environment_map(scenario: CampusScenario) -> None:
    """绘制单个场景的校园环境图，突出需求热点、遮挡区和禁装区。

    参数：
    - scenario：当前校园 WiFi 场景。

    返回：
    - 无。函数直接显示环境图并打印区域说明。
    """
    fig, ax = plt.subplots(figsize=(9, 6.8))
    ax.set_title(f"{scenario.name} 校园环境与 WiFi 禁装区域")
    ax.set_xlim(0, scenario.width)
    ax.set_ylim(0, scenario.height)
    ax.set_xlabel("校园平面 x 坐标")
    ax.set_ylabel("校园平面 y 坐标")
    ax.grid(True, linestyle="--", alpha=0.28)

    for obstacle in scenario.obstacles:
        if obstacle.no_install:
            facecolor = "#ffcdd2"
            edgecolor = "#b71c1c"
            hatch = "///"
        else:
            facecolor = "#c8e6c9"
            edgecolor = "#2e7d32"
            hatch = ""

        rect = plt.Rectangle(
            (obstacle.x1, obstacle.y1),
            obstacle.x2 - obstacle.x1,
            obstacle.y2 - obstacle.y1,
            facecolor=facecolor,
            edgecolor=edgecolor,
            hatch=hatch,
            linewidth=1.8,
            alpha=0.75,
            zorder=1,
        )
        ax.add_patch(rect)

        center_x = (obstacle.x1 + obstacle.x2) / 2
        center_y = (obstacle.y1 + obstacle.y2) / 2
        label = f"{obstacle.name}\n衰减 {obstacle.loss:.0f} dB"
        if obstacle.no_install:
            label += "\n禁止安装"
        else:
            label += "\n可安装"
        ax.text(center_x, center_y, label, ha="center", va="center", fontsize=8, color="#3e2723")

    for point in scenario.demands:
        size = 85 + point.weight * 70
        ax.scatter(point.x, point.y, s=size, color="#1565c0", edgecolor="white", linewidth=1.2, zorder=3)
        ax.text(
            point.x + 1.4,
            point.y + 1.4,
            f"{point.name}\n需求权重 {point.weight}",
            fontsize=8,
            color="#0d47a1",
        )

    legend_items = [
        Line2D([0], [0], marker="o", color="w", label="WiFi 需求热点", markerfacecolor="#1565c0", markersize=10),
        Patch(facecolor="#ffcdd2", edgecolor="#b71c1c", hatch="///", label="无法安装 WiFi 的区域"),
        Patch(facecolor="#c8e6c9", edgecolor="#2e7d32", label="可安装但有信号衰减的区域"),
    ]
    ax.legend(handles=legend_items, loc="upper right")
    plt.show()

    print(f"{scenario.name} 环境说明：")
    print("需求热点：")
    for point in scenario.demands:
        print(f"- {point.name}: 坐标=({point.x:.1f}, {point.y:.1f}), 需求权重={point.weight}")
    print("环境区域：")
    for obstacle in scenario.obstacles:
        print(
            f"- {obstacle.name}: 范围=({obstacle.x1:.1f}, {obstacle.y1:.1f}) 到 "
            f"({obstacle.x2:.1f}, {obstacle.y2:.1f}), 衰减={obstacle.loss:.0f} dB, "
            f"{describe_obstacle_type(obstacle)}"
        )


def plot_all_environment_maps(scenarios: List[CampusScenario]) -> None:
    """依次绘制所有校园场景的环境图。"""
    for scenario in scenarios:
        plot_environment_map(scenario)


plot_environment_map(scenario)

## 4. 粒子群算法求解路由器位置（学生补全）

前面的程序已经完成了教学区场景、需求点、禁装区域、信号衰减模型和布置方案评价函数 `evaluate_layout()`。从这里开始，学生需要补全粒子群算法的核心过程。

在本实验中，一个粒子表示一组路由器坐标。如果教学区需要部署 `k` 个路由器，那么粒子的位置向量为：

`[x1, y1, x2, y2, ..., xk, yk]`

其中 `(x1, y1)` 是第 1 个路由器坐标，`(x2, y2)` 是第 2 个路由器坐标。

完成粒子群算法求解时，可以按下面顺序进行：

1. **理解已有目标函数**：先阅读 `evaluate_layout(vector, scenario)`，明确它会返回一个综合代价 `cost`，代价越小表示布置越好。
2. **确定粒子表示方法**：确认 `self.dim = scenario.router_count * 2`，每两个连续数值表示一个路由器的 `(x, y)` 坐标。
3. **补全随机初始化**：在 `random_position()` 中随机生成路由器坐标；在 `random_velocity()` 中随机生成初始速度。
4. **补全粒子群初始化**：在 `initialize()` 中生成粒子群，计算每个粒子的初始代价，记录个体最优和全局最优。
5. **补全速度与位置更新**：在 `update_particle(index)` 中使用粒子群公式更新速度和位置。
6. **补全主循环**：在 `run()` 中反复更新所有粒子，评价新位置，更新个体最优和全局最优。
7. **观察优化结果**：补全后运行最后一个单元，查看教学区最优路由器位置、覆盖率、平均信号和收敛曲线。

建议补全顺序：`random_position()` -> `random_velocity()` -> `initialize()` -> `update_particle()` -> `run()`。

In [ ]:
def distance_xy(a: Tuple[float, float], b: Tuple[float, float]) -> float:
    """计算两个坐标点之间的欧氏距离。"""
    return math.hypot(a[0] - b[0], a[1] - b[1])


def segment_crosses_obstacle(a: Tuple[float, float], b: Tuple[float, float], obstacle: Obstacle) -> bool:
    """用采样方式近似判断路由器到需求点的连线是否穿过障碍物。"""
    samples = 30
    for t in range(samples + 1):
        ratio = t / samples
        x = a[0] + (b[0] - a[0]) * ratio
        y = a[1] + (b[1] - a[1]) * ratio
        if obstacle.contains(x, y):
            return True
    return False


def router_positions_from_vector(vector: List[float]) -> List[Tuple[float, float]]:
    """把粒子位置向量转换为路由器坐标列表。"""
    return [(vector[i], vector[i + 1]) for i in range(0, len(vector), 2)]


def signal_strength(router: Tuple[float, float], demand: DemandPoint, scenario: CampusScenario) -> float:
    """估计某个路由器到某个需求点的信号强度。

    这里使用简化模型：
    - 距离越远，信号越弱。
    - 连线穿过障碍物时，按障碍物 loss 增加额外衰减。
    """
    d = max(distance_xy(router, (demand.x, demand.y)), 1.0)
    signal = -28.0 - 22.0 * math.log10(d)
    for obstacle in scenario.obstacles:
        if segment_crosses_obstacle(router, (demand.x, demand.y), obstacle):
            signal -= obstacle.loss
    return signal


def router_inside_forbidden_area(router: Tuple[float, float], scenario: CampusScenario) -> bool:
    """判断路由器是否落入禁装区域。"""
    x, y = router
    if not (0 <= x <= scenario.width and 0 <= y <= scenario.height):
        return True
    return any(ob.no_install and ob.contains(x, y) for ob in scenario.obstacles)


def evaluate_layout(vector: List[float], scenario: CampusScenario) -> Tuple[float, Dict[str, float]]:
    """评价一个路由器布置方案。

    参数：
    - vector：粒子位置向量，包含所有路由器坐标。
    - scenario：当前校园场景。

    返回：
    - cost：综合代价，越小越好。
    - metrics：覆盖率、平均信号等指标，便于观察结果。
    """
    routers = router_positions_from_vector(vector)
    total_weight = sum(point.weight for point in scenario.demands)
    weak_signal_penalty = 0.0
    uncovered_weight = 0.0
    weighted_signal_sum = 0.0
    router_loads = [0.0 for _ in routers]

    for point in scenario.demands:
        signals = [signal_strength(router, point, scenario) for router in routers]
        best_index = max(range(len(signals)), key=lambda idx: signals[idx])
        best_signal = signals[best_index]
        weighted_signal_sum += point.weight * best_signal
        router_loads[best_index] += point.weight

        if best_signal < scenario.signal_threshold:
            gap = scenario.signal_threshold - best_signal
            weak_signal_penalty += point.weight * gap * gap
            uncovered_weight += point.weight

    interference_penalty = 0.0
    for i in range(len(routers)):
        for j in range(i + 1, len(routers)):
            d = distance_xy(routers[i], routers[j])
            if d < scenario.min_router_distance:
                interference_penalty += (scenario.min_router_distance - d) ** 2

    forbidden_penalty = 0.0
    for router in routers:
        if router_inside_forbidden_area(router, scenario):
            forbidden_penalty += 1000.0

    average_load = total_weight / len(routers)
    load_balance_penalty = sum((load - average_load) ** 2 for load in router_loads)

    coverage_rate = 1.0 - uncovered_weight / total_weight
    average_signal = weighted_signal_sum / total_weight
    cost = (
        weak_signal_penalty
        + 0.8 * interference_penalty
        + forbidden_penalty
        + 8.0 * load_balance_penalty
    )

    metrics = {
        "coverage_rate": coverage_rate,
        "average_signal": average_signal,
        "weak_signal_penalty": weak_signal_penalty,
        "interference_penalty": interference_penalty,
        "forbidden_penalty": forbidden_penalty,
        "load_balance_penalty": load_balance_penalty,
    }
    return cost, metrics

In [ ]:
class ParticleSwarmOptimizer:
    """粒子群算法优化器。

    本练习版保留了类结构和参数说明，但删除了粒子群算法的关键实现。
    请根据 TODO 注释补全。
    """

    def __init__(
        self,
        scenario: CampusScenario,
        num_particles: int = 45,
        num_iterations: int = 120,
        inertia_weight: float = 0.72,
        cognitive_weight: float = 1.45,
        social_weight: float = 1.45,
        max_velocity: float = 12.0,
    ):
        """初始化粒子群参数。

        参数：
        - scenario：待优化的校园 WiFi 场景。
        - num_particles：粒子数量。
        - num_iterations：迭代轮数。
        - inertia_weight：惯性权重，控制粒子保持原速度的程度。
        - cognitive_weight：个体学习因子，控制粒子靠近自身历史最好位置的程度。
        - social_weight：群体学习因子，控制粒子靠近全局最好位置的程度。
        - max_velocity：速度上限，防止粒子一步移动过远。
        """
        self.scenario = scenario
        self.num_particles = num_particles
        self.num_iterations = num_iterations
        self.inertia_weight = inertia_weight
        self.cognitive_weight = cognitive_weight
        self.social_weight = social_weight
        self.max_velocity = max_velocity
        self.dim = scenario.router_count * 2
        self.positions: List[List[float]] = []
        self.velocities: List[List[float]] = []
        self.personal_best_positions: List[List[float]] = []
        self.personal_best_costs: List[float] = []
        self.global_best_position: List[float] = []
        self.global_best_cost = float("inf")
        self.history: List[float] = []

    def random_position(self) -> List[float]:
        """随机生成一个粒子位置，即一组路由器坐标。

        参数：
        - 无。函数使用 `self.scenario.width`、`self.scenario.height` 和路由器数量。

        返回：
        - 长度为 `self.dim` 的列表，例如 `[x1, y1, x2, y2, ...]`。
        """
        # TODO 1：创建空列表 vector。
        # TODO 2：循环 self.scenario.router_count 次。
        # TODO 3：每次随机生成一个 x 坐标，范围为 [0, self.scenario.width]。
        # TODO 4：每次随机生成一个 y 坐标，范围为 [0, self.scenario.height]。
        # TODO 5：把 x 和 y 依次加入 vector，并返回 vector。
        raise NotImplementedError("请补全 random_position 方法")

    def random_velocity(self) -> List[float]:
        """随机生成粒子初始速度。

        参数：
        - 无。函数使用 `self.dim` 和 `self.max_velocity`。

        返回：
        - 长度为 `self.dim` 的速度列表。
        """
        # TODO：为每个维度随机生成一个速度，范围为 [-self.max_velocity, self.max_velocity]。
        raise NotImplementedError("请补全 random_velocity 方法")

    def clip_position(self, vector: List[float]) -> List[float]:
        """把粒子坐标限制在校园区域边界内。

        参数：
        - vector：粒子位置向量。

        返回：
        - 裁剪后的粒子位置向量，保证所有路由器坐标仍在教学区范围内。
        """
        clipped = vector[:]
        for i in range(0, len(clipped), 2):
            clipped[i] = min(max(clipped[i], 0.0), self.scenario.width)
            clipped[i + 1] = min(max(clipped[i + 1], 0.0), self.scenario.height)
        return clipped

    def initialize(self) -> None:
        """初始化粒子群，并记录初始个体最优和全局最优。

        参数：
        - 无。函数使用初始化时设置的粒子数量和场景。

        返回：
        - 无。函数会更新 positions、velocities、personal_best 和 global_best。
        """
        # TODO 1：循环 self.num_particles 次，生成每个粒子的初始 position 和 velocity。
        # TODO 2：调用 evaluate_layout(position, self.scenario) 计算该位置的代价 cost。
        # TODO 3：把 position、velocity 保存到 self.positions 和 self.velocities。
        # TODO 4：初始时每个粒子的当前位置就是它自己的个体最优位置。
        # TODO 5：如果当前 cost 小于 self.global_best_cost，则更新全局最优位置和全局最优代价。
        raise NotImplementedError("请补全 initialize 方法")

    def update_particle(self, index: int) -> None:
        """更新单个粒子的速度和位置。

        参数：
        - index：要更新的粒子编号。

        返回：
        - 无。函数会直接更新 self.positions[index] 和 self.velocities[index]。
        """
        # TODO 1：取出当前粒子的 position、velocity 和 personal_best。
        # TODO 2：对每一个维度 d，生成两个随机数 r1、r2。
        # TODO 3：计算个体学习项：
        #         cognitive_weight * r1 * (personal_best[d] - position[d])
        # TODO 4：计算群体学习项：
        #         social_weight * r2 * (global_best_position[d] - position[d])
        # TODO 5：根据公式更新速度：
        #         new_velocity = inertia_weight * velocity[d] + cognitive + social
        # TODO 6：把速度限制在 [-max_velocity, max_velocity]。
        # TODO 7：用新速度更新粒子位置，并调用 clip_position 限制边界。
        raise NotImplementedError("请补全 update_particle 方法")

    def run(self) -> Tuple[List[float], float, List[float]]:
        """执行粒子群优化。

        参数：
        - 无。函数使用初始化时设置的粒子数量、迭代轮数和学习参数。

        返回：
        - global_best_position：搜索到的最优路由器坐标向量。
        - global_best_cost：最优布置方案的综合代价。
        - history：每轮迭代结束后的历史最优代价列表。
        """
        # TODO 1：调用 initialize() 初始化粒子群。
        # TODO 2：进行 self.num_iterations 轮迭代。
        # TODO 3：每轮中依次更新每个粒子的位置。
        # TODO 4：调用 evaluate_layout 计算更新后位置的代价。
        # TODO 5：如果新代价优于该粒子的个体最优，则更新 personal_best。
        # TODO 6：如果新代价优于全局最优，则更新 global_best。
        # TODO 7：每轮结束后，把 self.global_best_cost 加入 self.history。
        # TODO 8：按示例格式打印部分迭代轮次的当前最优代价。
        # TODO 9：返回 global_best_position、global_best_cost 和 history。
        raise NotImplementedError("请补全 run 方法")

In [ ]:
def plot_solution(scenario: CampusScenario, best_vector: List[float], history: List[float]) -> None:
    """绘制某个场景下的最优路由器布置和收敛曲线。"""
    routers = router_positions_from_vector(best_vector)
    cost, metrics = evaluate_layout(best_vector, scenario)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.8), constrained_layout=True)
    ax = axes[0]
    ax.set_title(f"{scenario.name} 最优 WiFi 路由器布置")
    ax.set_xlim(0, scenario.width)
    ax.set_ylim(0, scenario.height)
    ax.set_xlabel("校园平面 x 坐标")
    ax.set_ylabel("校园平面 y 坐标")
    ax.grid(True, linestyle="--", alpha=0.3)

    for obstacle in scenario.obstacles:
        color = "#d9b38c" if obstacle.no_install else "#b5d6a7"
        rect = plt.Rectangle(
            (obstacle.x1, obstacle.y1),
            obstacle.x2 - obstacle.x1,
            obstacle.y2 - obstacle.y1,
            facecolor=color,
            edgecolor="#6d4c41",
            alpha=0.5,
        )
        ax.add_patch(rect)

    for point in scenario.demands:
        signals = [signal_strength(router, point, scenario) for router in routers]
        best_signal = max(signals)
        color = "#2e7d32" if best_signal >= scenario.signal_threshold else "#c62828"
        ax.scatter(point.x, point.y, s=70 + point.weight * 55, color=color, edgecolor="white", zorder=3)
        ax.text(point.x + 1.2, point.y + 1.2, f"{point.name}\\n{best_signal:.1f} dBm", fontsize=8)

    for idx, router in enumerate(routers, 1):
        ax.scatter(router[0], router[1], s=240, marker="*", color="#f9a825", edgecolor="#4e342e", zorder=4)
        circle = plt.Circle(router, 18, fill=False, linestyle="--", color="#f9a825", alpha=0.55)
        ax.add_patch(circle)
        ax.text(router[0] + 1.5, router[1] + 1.5, f"R{idx}", fontsize=11, weight="bold")

    ax.text(
        0.02,
        0.02,
        f"覆盖率：{metrics['coverage_rate'] * 100:.1f}%\\n平均信号：{metrics['average_signal']:.1f} dBm\\n总代价：{cost:.2f}",
        transform=ax.transAxes,
        bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#999999"},
    )

    axes[1].plot(range(1, len(history) + 1), history, color="#00695c", linewidth=2)
    axes[1].set_title(f"{scenario.name} 粒子群算法收敛曲线")
    axes[1].set_xlabel("迭代轮数")
    axes[1].set_ylabel("历史最优代价")
    axes[1].grid(True, linestyle="--", alpha=0.35)
    plt.show()

    print(f"{scenario.name} 最优路由器坐标：")
    for idx, router in enumerate(routers, 1):
        print(f"R{idx}: ({router[0]:.2f}, {router[1]:.2f})")
    print(f"覆盖率：{metrics['coverage_rate'] * 100:.1f}%")
    print(f"平均信号：{metrics['average_signal']:.1f} dBm")
    print(f"综合代价：{cost:.2f}")


print("=" * 70)
optimizer = ParticleSwarmOptimizer(
    scenario,
    num_particles=50,
    num_iterations=140,
    inertia_weight=0.72,
    cognitive_weight=1.45,
    social_weight=1.45,
    max_velocity=10.0,
)
best_vector, best_cost, history = optimizer.run()
results = {scenario.name: (best_vector, best_cost, history)}
plot_solution(scenario, best_vector, history)

## 5. 实验思考

完成实验后，可以继续思考下面的问题：

1. 如果增加路由器数量，覆盖率一定会提高吗？干扰是否也会增加？
2. 如果把 `min_router_distance` 调大，路由器会不会被迫分散？对覆盖率有什么影响？
3. 如果某个区域用户权重变大，粒子群算法找到的路由器位置是否会向该区域移动？
4. 本实验使用的是简化信号模型。真实校园部署中，还需要考虑哪些因素？
5. 粒子群算法和遗传算法都可以优化连续位置，它们在搜索方式上有什么不同？